# Qwen geometry + generation calibration (полный Colab run)

Этот notebook запускает **полный калибровочный прогон без `limit` и без `case_name`** для `Qwen/Qwen2.5-1.5B` через `C:/Users/Kharki/Desktop/ABPT/scripts/run_qwen_geometry_generation_calibration.py`.

Что делает прогон:
- один geometry forward на слоях `18..27` для каждого anchor-кейса
- полная base generation
- полная anchor-biased generation
- keyword/constraint analysis
- запись итогов в JSON и Markdown

Это именно **full run harness для Google Colab GPU**, а не chunked smoke-test.


> Рекомендуемый runtime: **GPU (T4 / L4 / A100)**.
>
> На CPU запуск тоже возможен, но будет очень медленным.
>
> По умолчанию notebook гоняет **все кейсы целиком**.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q torch transformers einops pytest numpy


In [ ]:
%cd /content
import os

if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
%cd /content/ABPT
!git fetch --all
!git checkout main
!git pull --ff-only origin main


In [ ]:
import json
import shlex
import subprocess
from pathlib import Path

import torch

from src.data.qwen_anchor_geometry_cases import make_qwen_anchor_geometry_cases

MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_LENGTH = 128
MAX_NEW_TOKENS = 120
SEED = 7

OUTPUT_JSON = '/content/ABPT/archive/qwen_geometry_generation_calibration.json'
OUTPUT_MD = '/content/ABPT/docs/research/qwen_geometry_generation_calibration.md'

cases = make_qwen_anchor_geometry_cases()
print('device =', DEVICE)
print('n_cases =', len(cases))
print('case names =', [case.name for case in cases])
print('output_json =', OUTPUT_JSON)
print('output_md =', OUTPUT_MD)


In [ ]:
for case in make_qwen_anchor_geometry_cases():
    print(f'[{case.anchor_class}] {case.name}')
    print(' anchor_group:', case.anchor_group)
    print(' anchor_text :', case.anchor_text)
    print(' prompt      :', case.prompt)
    print()


In [ ]:
cmd = [
    'python',
    '-u',
    'scripts/run_qwen_geometry_generation_calibration.py',
    '--model', MODEL_NAME,
    '--device', DEVICE,
    '--max_length', str(MAX_LENGTH),
    '--max_new_tokens', str(MAX_NEW_TOKENS),
    '--seed', str(SEED),
    '--output_json', OUTPUT_JSON,
    '--output_md', OUTPUT_MD,
]
print('Running full calibration command:')
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)


In [ ]:
payload = json.loads(Path(OUTPUT_JSON).read_text(encoding='utf-8'))
print('metadata:')
for key, value in payload['metadata'].items():
    print(f'  {key}: {value}')

print()
print('calibration summary:')
for cluster, stats in payload['calibration']['by_cluster'].items():
    print(cluster, stats)
print('threshold_candidates:', payload['calibration']['threshold_candidates'])

print()
print('per-case summary:')
for case in payload['cases']:
    print(
        f"{case['name']}: cluster={case['anchor_cluster']} | "
        f"r1@24={case['r1_at_24']} | d26->27={case['delta_l26_l27']} | "
        f"slope={case['slope_l18_l24']} | "
        f"base_q={case['base_analysis']['quality_score']} | "
        f"anchor_q={case['anchor_analysis']['quality_score']} | "
        f"delta={case['constraint_delta']} | "
        f"drift={case['anchor_analysis']['drift_detected']}"
    )


In [ ]:
report_text = Path(OUTPUT_MD).read_text(encoding='utf-8')
print(report_text)


In [ ]:
# Optional: download artifacts from Colab runtime
# from google.colab import files
# files.download(OUTPUT_JSON)
# files.download(OUTPUT_MD)
